# AcneScope: máy chủ model trên Colab (GPU) + ngrok

Backend trên Vercel gọi vào đây để đếm mụn. **Chạy lần lượt từng ô, giữ tab mở** (Colab tắt khi rảnh lâu hoặc sau khoảng 12 giờ).

**Chuẩn bị một lần:**
1. `Runtime → Change runtime type → T4 GPU`.
2. Trên máy: `cd backend && python pack_colab.py`, rồi tải `backend/colab_bundle.zip` lên Google Drive vào thư mục `MyDrive/acnescope/`.
3. Mở biểu tượng 🔑 **Secrets** (thanh bên trái) và thêm, bật *Notebook access*:
   - `NGROK_AUTHTOKEN`: lấy ở https://dashboard.ngrok.com/get-started/your-authtoken
   - `INFER_API_KEY`: một chuỗi bí mật tự đặt, **trùng** với `INFERENCE_API_KEY` trên Vercel
   - `NGROK_DOMAIN` (khuyên dùng): domain tĩnh miễn phí ở https://dashboard.ngrok.com/domains, ví dụ `ten-ban.ngrok-free.app`. Có domain tĩnh thì địa chỉ không đổi giữa các lần chạy, không phải sửa Vercel.

⚠️ Ảnh khuôn mặt sẽ đi qua máy chủ Google Colab và ngrok. Code không lưu ảnh.

In [ ]:
# 1) Lấy code + model từ Drive
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/acnescope && mkdir -p /content/acnescope
!unzip -q -o /content/drive/MyDrive/acnescope/colab_bundle.zip -d /content/acnescope
!ls -la /content/acnescope /content/acnescope/app

In [ ]:
# 2) Cài thư viện (Colab đã có torch CUDA, KHÔNG cài lại torch)
!pip install -q ultralytics==8.4.192 pillow-heif pyngrok "pydantic-settings>=2.2" python-multipart httpx "uvicorn[standard]"
import torch, ultralytics
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU, kiểm tra Runtime type')
print('ultralytics', ultralytics.__version__)

In [ ]:
# 3) Chạy máy chủ model (nền) ở cổng 8001
import os, subprocess, time, httpx
from google.colab import userdata

API_KEY = userdata.get('INFER_API_KEY')
env = {**os.environ, 'INFER_API_KEY': API_KEY, 'MODEL_WEIGHTS': 'best.pt',
       # Cấu hình model, giữ giống backend/.env (đổi ở đây nếu cần):
       'MODEL_IMGSZ': '1280', 'MODEL_END2END': 'true', 'MODEL_HALF': 'true', 'MODEL_MAX_DET': '500',
       'CONF_FLOOR': '0.05', 'DEDUP_IOU': '0.3', 'DEDUP_CONTAIN': '0.7'}
try: server.kill()
except NameError: pass
log = open('/content/infer.log', 'w')
server = subprocess.Popen(['python', '-m', 'uvicorn', 'infer_server:app', '--host', '0.0.0.0', '--port', '8001'],
                          cwd='/content/acnescope', env=env, stdout=log, stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        r = httpx.get('http://127.0.0.1:8001/health', headers={'X-API-Key': API_KEY}, timeout=2)
        if r.status_code == 200: print('Máy chủ model OK:', r.json()); break
    except Exception: pass
    time.sleep(1)
else:
    print('Chưa chạy được, xem log:'); print(open('/content/infer.log').read()[-3000:])

In [ ]:
# 4) Mở ra internet bằng ngrok
from pyngrok import ngrok
ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
try: DOMAIN = userdata.get('NGROK_DOMAIN')
except Exception: DOMAIN = None
ngrok.kill()
tunnel = ngrok.connect(8001, 'http', domain=DOMAIN) if DOMAIN else ngrok.connect(8001, 'http')
PUBLIC_URL = tunnel.public_url
r = httpx.get(PUBLIC_URL + '/health', headers={'X-API-Key': API_KEY, 'ngrok-skip-browser-warning': '1'}, timeout=20)
print('Kiểm tra qua ngrok:', r.status_code, r.text[:200])
print('\n==> Đặt trên Vercel (project backend):  INFERENCE_URL =', PUBLIC_URL)
if not DOMAIN: print('(Không có NGROK_DOMAIN: địa chỉ này ĐỔI mỗi lần chạy, phải sửa INFERENCE_URL trên Vercel rồi redeploy.)')

In [ ]:
# 5) (Tuỳ chọn) Xem log máy chủ model
print(open('/content/infer.log').read()[-3000:])